# CareerLens — Skills Extraction Pipeline v2
Advanced TF-IDF pipeline with synonym-based role mapping, regex text pre-processing, and custom domain stop words.

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text as sk_text

warnings.filterwarnings("ignore")
print("All libraries loaded successfully.")


## Step 1: Data Standardization & Merging

In [ ]:
print("[Step 1] Loading and standardizing datasets...")

# 1a. all_job_post.csv
df_all_job = pd.read_csv("all_job_post.csv")
df_all_job = df_all_job[["job_title", "job_skill_set"]].rename(
    columns={"job_title": "role", "job_skill_set": "skills"}
)

# 1b. job_dataset.csv
df_job_dataset = pd.read_csv("job_dataset.csv")
df_job_dataset = df_job_dataset[["Title", "Skills"]].rename(
    columns={"Title": "role", "Skills": "skills"}
)

# 1c. final_data.csv — one-hot encoded skills
df_final = pd.read_csv("final_data.csv")
skill_cols = df_final.columns[10:]

def onehot_to_skills(row):
    return ", ".join([col for col in skill_cols if row[col] == 1 or row[col] is True])

df_final_extracted = pd.DataFrame({
    "role": df_final["Designation"],
    "skills": df_final.apply(onehot_to_skills, axis=1)
})

# Concatenate and clean
master_df = pd.concat([df_all_job, df_job_dataset, df_final_extracted], ignore_index=True)
master_df = master_df.dropna(subset=["role", "skills"]).reset_index(drop=True)
master_df["skills"] = master_df["skills"].astype(str).str.strip()
master_df = master_df[master_df["skills"] != ""].reset_index(drop=True)

print(f"  Master DataFrame: {len(master_df)} rows after merging and cleaning.")


## Step 2: Advanced Role Mapping (Synonym/Alias Dictionary)

In [ ]:
print("[Step 2] Mapping roles with synonym dictionary...")

roles_df = pd.read_csv("roles.csv")

# ── Synonym / Alias Dictionary ───────────────────────────────────────────
# Maps role_id -> list of alias keywords to match against job titles.
synonym_dict = {
    "R001": ["software engineer", "software developer", "programmer", "swe"],
    "R002": ["frontend", "front end", "front-end", "react developer", "vue developer", "angular developer", "ui developer"],
    "R003": ["backend", "back end", "back-end", "api developer", "server-side developer"],
    "R004": ["full stack", "fullstack", "full-stack"],
    "R005": ["mobile developer", "android developer", "ios developer", "flutter developer", "react native developer", "app developer"],
    "R006": ["devops", "dev ops", "site reliability", "sre engineer", "platform engineer", "devsecops"],
    "R007": ["qa engineer", "quality assurance", "software tester", "test engineer", "qa analyst"],
    "R008": ["system analyst", "systems analyst", "business systems analyst"],
    "R009": ["game developer", "game engineer", "game programmer"],
    "R010": ["embedded systems", "firmware engineer", "rtos engineer", "iot engineer"],
    "R011": ["cloud engineer", "cloud architect", "aws engineer", "azure engineer", "gcp engineer"],
    "R012": ["system administrator", "sysadmin", "it administrator", "linux administrator"],
    "R013": ["network engineer", "network administrator", "network analyst"],
    "R014": ["project manager", "program manager", "it manager", "delivery manager", "scrum master"],
    "R015": ["augmented reality", "virtual reality", "metaverse developer", "xr developer"],
    "R016": ["blockchain developer", "web3 developer", "solidity developer", "smart contract developer"],
    "R017": ["systems engineer", "system engineer", "solutions engineer"],
    "R018": ["infrastructure architect", "infrastructure engineer"],
    "R019": ["virtualization specialist", "vmware specialist"],
    "R020": ["cybersecurity analyst", "cyber security analyst", "security architect", "information security analyst"],
    "R021": ["security consultant", "it security consultant"],
    "R022": ["ethical hacker", "penetration tester", "pentester", "red team engineer"],
    "R023": ["soc analyst", "security operations analyst"],
    "R024": ["information security specialist", "infosec specialist"],
    "R025": ["cyber defense analyst", "cyber defence analyst"],
    "R026": ["solutions architect", "solution architect", "enterprise architect"],
    "R027": ["data analyst", "analytics analyst", "reporting analyst"],
    "R028": ["data scientist", "research scientist"],
    "R029": ["machine learning engineer", "ml engineer"],
    "R030": ["ai engineer", "artificial intelligence engineer"],
    "R031": ["business intelligence analyst", "bi analyst", "bi developer", "bi engineer"],
    "R032": ["data engineer", "etl developer", "etl engineer"],
    "R033": ["product analyst", "growth analyst"],
    "R034": ["big data engineer", "hadoop engineer", "spark engineer"],
    "R035": ["analytics engineer", "dbt engineer"],
    "R036": ["deep learning engineer", "neural network engineer"],
    "R037": ["nlp engineer", "natural language processing engineer", "computational linguist"],
    "R038": ["computer vision engineer", "cv engineer", "image processing engineer"],
    "R039": ["prompt engineer", "llm engineer", "generative ai engineer"],
    "R040": ["data modeler", "data modelling specialist"],
    "R041": ["data architect"],
    "R042": ["database developer", "db developer"],
    "R043": ["database administrator", "dba"],
    "R044": ["ui designer", "user interface designer"],
    "R045": ["ux designer", "user experience designer"],
    "R046": ["product designer"],
    "R047": ["graphic designer", "visual designer"],
    "R048": ["motion designer", "motion graphics designer", "animator"],
    "R049": ["ux researcher", "user researcher", "usability researcher"],
    "R050": ["illustrator", "digital illustrator"],
    "R051": ["3d designer", "3d artist", "3d modeler"],
    "R052": ["game ui designer", "game interface designer"],
    "R053": ["design system designer"],
    "R054": ["unity developer", "unreal developer", "arvr developer"],
    "R055": ["digital marketing specialist", "online marketing specialist", "growth marketer"],
    "R056": ["seo specialist", "search engine optimization specialist"],
    "R057": ["content strategist"],
    "R058": ["performance marketer", "paid media specialist", "google ads specialist", "ppc specialist"],
    "R059": ["marketing analyst"],
    "R060": ["crm specialist", "salesforce specialist", "hubspot specialist"],
    "R061": ["content writer", "copywriter"],
    "R062": ["social media manager", "social media specialist"],
    "R063": ["community manager"],
    "R064": ["virtual assistant"],
    "R065": ["marketing associate", "marketing coordinator"],
    "R066": ["ecommerce manager", "e-commerce manager"],
    "R067": ["campaign analyst", "campaign manager"],
    "R068": ["business analyst"],
}

# Pre-compute cleaned role names from roles.csv
def clean_role_name(name):
    return re.split(r"[/(]", str(name).lower())[0].strip()

roles_df["_clean"] = roles_df["role_name"].apply(clean_role_name)

# Build combined lookup: alias -> role_id (longer phrases take priority)
alias_to_id = {}
for rid, aliases in synonym_dict.items():
    for alias in aliases:
        alias_to_id[alias] = rid
for _, row in roles_df.iterrows():
    alias_to_id[row["_clean"]] = row["role_id"]

# Sort by descending alias length so longer, more specific phrases match first
sorted_aliases = sorted(alias_to_id.keys(), key=len, reverse=True)

def map_to_role_id(role_str):
    s = str(role_str).lower().strip()
    for alias in sorted_aliases:
        if alias in s:
            return alias_to_id[alias]
    return None

master_df["mapped_role_id"] = master_df["role"].apply(map_to_role_id)
master_filtered = master_df.dropna(subset=["mapped_role_id"]).reset_index(drop=True)

print(f"  Total mapped rows  : {len(master_filtered)}")
print(f"  Unique roles found : {master_filtered['mapped_role_id'].nunique()} / 68")
print(f"  Empty roles        : {68 - master_filtered['mapped_role_id'].nunique()}")


## Step 3: Text Pre-processing (Regex) & Custom Stop Words

In [ ]:
print("[Step 3] Applying regex cleaning and building custom stop words...")

def clean_skills_text(text):
    t = str(text).lower()
    # Consolidate compound tech terms before tokenisation
    t = re.sub(r"node\.?js", "nodejs", t)
    t = re.sub(r"vue\.?js", "vuejs", t)
    t = re.sub(r"react\.?js", "reactjs", t)
    t = re.sub(r"next\.?js", "nextjs", t)
    t = re.sub(r"express\.?js", "expressjs", t)
    t = re.sub(r"angular\.?js", "angularjs", t)
    t = re.sub(r"three\.?js", "threejs", t)
    t = re.sub(r"ui\s*/\s*ux", "uiux", t)
    t = re.sub(r"ux\s*/\s*ui", "uiux", t)
    t = re.sub(r"c\+\+", "cpp", t)
    t = re.sub(r"c#", "csharp", t)
    t = re.sub(r"\.net\b", "dotnet", t)
    t = re.sub(r"a/b\s*test", "ab testing", t)
    # Strip remaining special chars, keep alphanumeric and spaces
    t = re.sub(r"[^\w\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

master_filtered = master_filtered.copy()
master_filtered["skills_clean"] = master_filtered["skills"].apply(clean_skills_text)

# Build combined stop words list
domain_stop_words = [
    "ai", "ui", "ux", "js", "basics", "tools", "skills", "experience",
    "using", "development", "knowledge", "understanding", "ability",
    "working", "good", "strong", "proficiency", "proficient", "expert",
    "advanced", "intermediate", "awareness", "familiarity",
    "learn", "tool", "support", "management", "software", "use",
    "including", "related", "web", "based", "systems"
]

combined_stop_words = list(sk_text.ENGLISH_STOP_WORDS.union(set(domain_stop_words)))
print(f"  Total stop words: {len(combined_stop_words)}")
print(f"  Sample cleaned  : {master_filtered['skills_clean'].iloc[0][:150]}")


## Step 4: TF-IDF Skill Extraction

In [ ]:
print("[Step 4] Running TF-IDF per role...")

# Aggregate cleaned skills into one corpus document per role
grouped = (
    master_filtered
    .groupby("mapped_role_id")["skills_clean"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
    .rename(columns={"mapped_role_id": "role_id", "skills_clean": "corpus"})
)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    stop_words=combined_stop_words,
    min_df=1
)

top_skills_records = []

for _, row in grouped.iterrows():
    rid = row["role_id"]
    corpus = row["corpus"]
    try:
        tfidf_matrix = vectorizer.fit_transform([corpus])
        features = vectorizer.get_feature_names_out()
        scores = tfidf_matrix.toarray()[0]
        top_idx = scores.argsort()[-10:][::-1]
        top_skills = [features[i].title() for i in top_idx]
        skill_str = ", ".join(top_skills)
    except ValueError:
        skill_str = None
    top_skills_records.append({"role_id": rid, "skill_rekomendasi_industri": skill_str})

top_skills_df = pd.DataFrame(top_skills_records)
print(f"  TF-IDF extraction done for {len(top_skills_df)} roles.")


## Step 5: Output and Saving

In [ ]:
print("[Step 5] Merging results and saving...")

final_output = pd.merge(
    roles_df.drop(columns=["_clean"], errors="ignore"),
    top_skills_df,
    on="role_id",
    how="left"
)

# Fill roles that got no match in the dataset
final_output["skill_rekomendasi_industri"] = (
    final_output["skill_rekomendasi_industri"]
    .fillna("Needs Manual Review")
)

# Resolve output directory (handles running from inside or outside bismillah/)
cwd = os.getcwd()
if os.path.basename(cwd).lower() == "bismillah":
    output_dir = cwd
else:
    output_dir = os.path.join(cwd, "bismillah")
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "master_roles_updated_with_skills_v2.csv")
final_output.to_csv(output_path, index=False)

# ── Summary ────────────────────────────────────────────────────────────────
filled   = (final_output["skill_rekomendasi_industri"] != "Needs Manual Review").sum()
unfilled = len(final_output) - filled

print(f"\n  Saved : {output_path}")
print(f"  Shape : {final_output.shape}")
print(f"  Roles with skills         : {filled} / {len(final_output)}")
print(f"  Roles needing manual review: {unfilled}")
print("\n--- PREVIEW ---")
print(final_output[["role_id", "role_name", "skill_rekomendasi_industri"]].to_string(index=False))
